In [ ]:
import pandas as pd
from pandas.core.groupby import groupby


def profile_df(df, name):
    print(f"\n{'=' * 40}")
    print(f"DATASET: {name}")
    print(f"{'=' * 40}")

    print(f"\nShape:\n"
          f"{df.shape}")

    print(f"\nColumns:\n"
          f"{df.columns.to_list()}")

    print(f"\nData type:\n"
          f"{df.dtypes}")

    print(f"\nMissing values:\n"
          f"{df.isnull().sum()}")

    print(f"\nDuplicated values:\n"
          f"{df.duplicated().sum()}")

    print(f"\nFirst 5 rows:\n"
          f"{df.head()}")


inventory_df = pd.read_csv("../data/raw/inventory_of_generation_2025.csv", sep="\t")
domestic_df = pd.read_csv("../data/raw/monthly_domestic_values_2025.csv", sep="\t")
hourly_load_df = pd.read_csv("../data/raw/monthly_hourly_load_values_2025.csv", sep="\t")
physical_energy_df = pd.read_csv("../data/raw/physical_energy_power_flows_2025.csv", sep="\t")


profile_df(hourly_load_df, "Monthly Hourly Load Values")
profile_df(physical_energy_df, "Physical Energy Power Flows")


# 1. Invesory of Generation

## 1.1 Overview

* How many records are in the dataset?
* What columns are available?
* What are the data types?
* Are there missing values?
* Are there duplicate records?

In [ ]:
profile_df(inventory_df, "Inventory of Generation")

## 1.2 Categories and Countries

* What categories are present?
* How frequently does each category occur?
* How many countries are represented?
* How many records does each country have?
* Which categories are available for each country?
* Is there more than one record for the same country and generation category?
* Why does every country contain an Error category?

In [ ]:
print(inventory_df['Category'].value_counts())

print(f"\n\n{inventory_df.groupby('Country')['Category'].value_counts()}")
# Is there more than one record for the same country and generation category?
print(f"\n\nDuplicate Country + Category combinations:\n{(inventory_df.groupby(['Country', 'Category']).size().loc[lambda x: x >=2])}")
# Countr + Category <= candidate key

print(f"\n\nThe Error category:\n{inventory_df.loc[inventory_df['Category'] == 'Error', 'Country'].value_counts()}")
# How many countries are represented in the dataset?
print(f"\n\nCountries:{inventory_df['Country'].nunique()}\n{inventory_df['Country'].unique()}")
# The dataset contains 36 countries, and each country has exactly one record with the Error category.

# How many records does each country have?
print(f"\n\nRecords:\n{inventory_df['Country'].value_counts()}")

# Which categories are available for each country?
inventory_df.groupby('Country')['Category'].agg(list)

## 1.3 ID

* What does MeasureItemID represent?
* What does MeasureItemCategoryID represent?
* Are these identifiers consistent across all records?
* Does each identifier correspond to exactly one logical entity?
* Does each MeasureItemID correspond to exactly one Category?

In [ ]:
print(f"{inventory_df['MeasureItemID'].value_counts()}")
print(f"\n\n{inventory_df['MeasureItemCategoryID'].value_counts()}")
# MeasureItemID appears to identify the generation category, while MeasureItemCategoryID is constant across the dataset.
print(f"\n\n{inventory_df.groupby('MeasureItemID')['Category'].nunique()}")
# MeasureItemID 1 : 1 Category
# MeasureItemID uniquely identifies a generation category, while MeasureItemCategoryID is constant across the dataset.
print(f"\n\n{inventory_df.groupby('Category')['MeasureItemID'].nunique()}")

# Why is MeasureItemCategoryID always equal to 9?
# MeasureItemCategoryID has a constant value of 9 across all records. Its meaning should be verified against the source metadata.

## 1.4 Time

* Does the dataset contain only 2025?

In [ ]:
print(f"{inventory_df['Year'].value_counts()}")
# All records refer to the year 2025.

## 1.5 Values

* What are the minimum, maximum, and average values?
* Are there zero or negative values?
* Is ProvidedValueCode populated?
* If it is empty, can it be safely excluded from the analytical layer?

In [ ]:
print(f"\n{'=' * 40}"
      f"\nProvidedValue"
      f"\n{'=' * 40}"
      f"\nMinimum: {inventory_df['ProvidedValue'].min()} "
      f"\nMaximum: {inventory_df['ProvidedValue'].max()} "
      f"\nAverage: {inventory_df['ProvidedValue'].mean():.2f}")

print(f"\n{'=' * 40}\n"
      f"{inventory_df['ProvidedValueCode'].isnull().value_counts()}")

print(f"\n{'=' * 40}"
      f"\nNumberOfUnits"
      f"\n{'=' * 40}"
      f"\nMinimum: {inventory_df['NumberOfUnits'].min()} "
      f"\nMaximum: {inventory_df['NumberOfUnits'].max()} "
      f"\nAverage: {inventory_df['NumberOfUnits'].mean():.2f}")

# NumberOfUnits ranges from 1 to 795, with an average of 35.31, and is generally much smaller than ProvidedValue.
# ProvidedValueCode is missing for all 211 records. Since the column contains no information in this dataset, it can be excluded from the analytical layer.
# The value-related fields contain no obvious data quality issues.

## 1.6 Conclusions

* Dataset contains 211 records.
* Grain: one record per Country + Category for 2025.
* Main dimensions: Country, Category.
* Main measures: ProvidedValue, NumberOfUnits.
* Retained: Country, Category, Year, MeasureItemID, MeasureItemCategoryID, ProvidedValue, and NumberOfUnits.
* Remove: ProvidedValueCode (empty), CreationDate, UpdateDate.
* Issues: Error is a systematic category present for every country and should not be removed without understanding its source meaning. MeasureItemCategoryID is constant (9) and its exact meaning should be verified from source metadata.

# 2. Monthly Domestic Values

## 2.1 Overview

* How many records are in the dataset?
* What columns are available?
* What are the data types?
* Are there missing values?
* Are there duplicate records?

In [ ]:
profile_df(domestic_df, "Monthly Domestic Values")

## 2.2 Categories and Areas

* What domestic categories are present?
* How many areas are represented?
* How many records does each category contain?
* How many records does each area contain?

In [ ]:
print(f"\n{'=' * 40}"
      f"\n{domestic_df['Category'].value_counts()}")

print(f"\n{'=' * 40}"
      f"\n{domestic_df['Category'].nunique()}\n{domestic_df['Category'].unique()}")

print(f"\n{'=' * 40}"
      f"\n{domestic_df['Area'].value_counts()}")

print(f"\n{'=' * 40}"
      f"\n{domestic_df['Area'].nunique()}\n{domestic_df['Area'].unique()}")

print(f"\n{'=' * 40}"
      f"\n{domestic_df.groupby('Area')['Category'].value_counts()}"
      f"\n{'=' * 40}"
      f"\n{domestic_df.groupby('Category')['Area'].value_counts()}")

# Does each Area + Category contain data for all 12 months?
months_per_group = (domestic_df.groupby(['Area', 'Category'])['Month'].nunique())

missing_months = (domestic_df.groupby(['Area', 'Category'])['Month'].apply(lambda x: sorted(set(range(1,13)) - set(x))))

print(f"\n{'=' * 40}"
      f"\n{months_per_group[months_per_group != 12]}"
      f"\n{'=' * 40}"
      f"\n{missing_months[missing_months.apply(len) > 0]}")

# Most Area + Category combinations contain all 12 months.
# 34 combinations have incomplete monthly coverage.
# Missing months vary by area and category, so they should be retained rather than treated as one general data quality issue.

## 2.3 ID

## 2.4 Time

## 2.5 Values

## 2.6 Conclusions

* Dataset contains  records.
* Grain:
* Main dimensions:
* Main measures:
* Retained:
* Remove:
* Issues: